# 04 Cascade Randomisation Experiment
Runs Adebayo et al.'s sanity check: progressively randomise the model top-down, regenerate saliency maps at each depth, and quantify similarity to the fully-trained baseline (Spearman, SSIM) + a random-vs-random null (Comparison B) to control for anatomy/architecture-driven similarity.

#### Setup

In [ ]:
# Import libraries, project modules (config, data, randomisation, saliency)
import pandas as pd
import torch
from scipy.stats import spearmanr
from skimage.metrics import structural_similarity as ssim

import src.config as config
from src.data import load_and_preprocess_image
from src.randomisation import cascade_randomise, get_top_down_layers
from src.saliency import (
    compute_vanilla_gradients, compute_guided_backprop,
    compute_gradcam, compute_integrated_gradients,
    compute_dataset_mean_baseline,
)

# Load the model and label set, and recompute the Integrated Gradients baseline
import torchxrayvision as xrv
model = xrv.models.DenseNet(weights="densenet121-res224-chex")
labels = pd.read_csv('../data/true_probability_labels.csv')
mean_baseline = compute_dataset_mean_baseline(labels['Image Index'].tolist(), '../data/xray_dataset')

# Confirm the number of top-down layers available for cascade randomisation
n_layers = len(get_top_down_layers(model))
print(f"Total layers available for randomisation: {n_layers}")

Total layers available for randomisation: 13


**Compute-budget decision:** given time constraints, this run uses 15 images and 5 randomisation depths - this is a documented limitation - the result is a smaller-N sanity check, not a full-scale reproduction.

In [ ]:
# Define experiment parameters (image count, randomisation depths, two independent seeds — one for the main trained-vs-randomised comparison, one for the random-vs-random null
N_IMAGES = 15
DEPTHS = sorted(set([0, n_layers // 4, n_layers // 2, (3 * n_layers) // 4, n_layers]))
SEED_A = config.SEED
SEED_B = config.SEED + 500  # second, independent seed for the null comparison

# Select the confidently-classified sample to test, and wrap all four saliency methods behind one function
experiment_images = labels[labels['True Label Probability'] >= 0.7].sample(
    n=N_IMAGES, random_state=config.SEED
)

def compute_all_methods(model, img, target_class):
    img = img.detach().clone().requires_grad_()
    return {
        'Vanilla Gradients': compute_vanilla_gradients(model, img, target_class),
        'Guided Backprop': compute_guided_backprop(model, img, target_class),
        'GradCAM': compute_gradcam(model, img, target_class, model.features.denseblock4),
        'Integrated Gradients': compute_integrated_gradients(model, img, target_class, mean_baseline, n_steps=20),
    }

results_rows = []

# For each image: compute baseline (fully-trained) saliency maps, then at each randomisation depth compute maps from a correspondingly randomised model and score their similarity to baseline (Spearman, SSIM)
# Separately, compute the random-vs-random null by comparing two independently-randomised (fully destroyed) models against each other
for _, row in experiment_images.iterrows():
    filename, pathology = row['Image Index'], row['Finding Labels']
    img = load_and_preprocess_image(f"../data/xray_dataset/{filename}")
    target_class = model.pathologies.index(pathology)

    baseline_maps = compute_all_methods(model, img, target_class)

    for depth in DEPTHS:
        maps_at_depth = baseline_maps if depth == 0 else compute_all_methods(
            cascade_randomise(model, depth, seed=SEED_A), img, target_class
        )
        for method_name in baseline_maps:
            corr, _ = spearmanr(baseline_maps[method_name].flatten(), maps_at_depth[method_name].flatten())
            s = ssim(baseline_maps[method_name], maps_at_depth[method_name], data_range=1.0)
            results_rows.append({
                'filename': filename, 'pathology': pathology, 'method': method_name,
                'comparison': 'trained_vs_randomised', 'depth': depth, 'spearman': corr, 'ssim': s,
            })

    # Comparison B: random-vs-random null, at full randomisation
    maps_a = compute_all_methods(cascade_randomise(model, n_layers, seed=SEED_A), img, target_class)
    maps_b = compute_all_methods(cascade_randomise(model, n_layers, seed=SEED_B), img, target_class)
    for method_name in baseline_maps:
        corr, _ = spearmanr(maps_a[method_name].flatten(), maps_b[method_name].flatten())
        s = ssim(maps_a[method_name], maps_b[method_name], data_range=1.0)
        results_rows.append({
            'filename': filename, 'pathology': pathology, 'method': method_name,
            'comparison': 'random_vs_random_null', 'depth': n_layers, 'spearman': corr, 'ssim': s,
        })

    print(f"Done: {filename} ({pathology})")

torch.Size([1, 1, 7, 7]) tensor(-0.0084, grad_fn=<MinBackward1>) tensor(0.0078, grad_fn=<MaxBackward1>)
torch.Size([1, 1, 7, 7]) tensor(-0.0009, grad_fn=<MinBackward1>) tensor(0.0002, grad_fn=<MaxBackward1>)
torch.Size([1, 1, 7, 7]) tensor(5.8315e-06, grad_fn=<MinBackward1>) tensor(0.0002, grad_fn=<MaxBackward1>)
torch.Size([1, 1, 7, 7]) tensor(-1.3548e-05, grad_fn=<MinBackward1>) tensor(-1.1521e-06, grad_fn=<MaxBackward1>)
torch.Size([1, 1, 7, 7]) tensor(-0.0048, grad_fn=<MinBackward1>) tensor(-0.0006, grad_fn=<MaxBackward1>)
torch.Size([1, 1, 7, 7]) tensor(-0.0048, grad_fn=<MinBackward1>) tensor(-0.0006, grad_fn=<MaxBackward1>)
torch.Size([1, 1, 7, 7]) tensor(8.5321e-05, grad_fn=<MinBackward1>) tensor(0.0054, grad_fn=<MaxBackward1>)
Done: 00014153_000.png (Effusion)
torch.Size([1, 1, 7, 7]) tensor(-0.0017, grad_fn=<MinBackward1>) tensor(0.0072, grad_fn=<MaxBackward1>)
torch.Size([1, 1, 7, 7]) tensor(-0.0008, grad_fn=<MinBackward1>) tensor(-1.2955e-05, grad_fn=<MaxBackward1>)
torch.Si

In [ ]:
# Ensure output directories exist, then save all raw similarity scores to `results/sanity_check_scores.csv` for analysis in the next notebook
from pathlib import Path
Path('../results').mkdir(exist_ok=True)
Path('../figures').mkdir(exist_ok=True)

In [ ]:
results_df = pd.DataFrame(results_rows)
results_df.to_csv('../results/sanity_check_scores.csv', index=False)
results_df.head()

,filename,pathology,method,comparison,depth,spearman,ssim
0,00014153_000.png,Effusion,Vanilla Gradients,trained_vs_randomised,0,1.000000,1.000000
1,00014153_000.png,Effusion,Guided Backprop,trained_vs_randomised,0,1.000000,1.000000
2,00014153_000.png,Effusion,GradCAM,trained_vs_randomised,0,1.000000,1.000000
3,00014153_000.png,Effusion,Integrated Gradients,trained_vs_randomised,0,1.000000,1.000000
4,00014153_000.png,Effusion,Vanilla Gradients,trained_vs_randomised,3,0.493608,0.547367
